In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

DATA_PATH = Path("../data/processed/cic_common.pkl")
feats = pd.read_pickle(DATA_PATH)

print("Dataset shape:", feats.shape)
print("\nClass counts:")
print(feats["family"].value_counts())

# Exclude target, metadata and target-derived label
X = feats.drop(
    columns=["family", "source_file", "is_attack"],
    errors="ignore"
)
y = feats["family"]

EXPECTED = [
    "dst_port", "duration_us", "fwd_pkts", "bwd_pkts",
    "fwd_bytes", "bwd_bytes", "bytes_per_s", "pkts_per_s",
    "syn_count", "rst_count", "fin_count"
]

assert list(X.columns) == EXPECTED, (
    f"Unexpected feature columns: {list(X.columns)}"
)
assert "is_attack" not in X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("\nFeature count:", X.shape[1])
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class counts:")
print(y_train.value_counts())

print("\nTesting class counts:")
print(y_test.value_counts())

Dataset shape: (2274686, 14)

Class counts:
family
benign        1860749
dos            313824
portscan        90618
bruteforce       7406
web              2089
Name: count, dtype: int64

Feature count: 11
Training rows: 1592280
Testing rows: 682406

Training class counts:
family
benign        1302524
dos            219677
portscan        63433
bruteforce       5184
web              1462
Name: count, dtype: int64

Testing class counts:
family
benign        558225
dos            94147
portscan       27185
bruteforce      2222
web              627
Name: count, dtype: int64


In [2]:
rf = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest training and prediction complete.")
print("Model feature count:", rf.n_features_in_)

Random Forest training and prediction complete.
Model feature count: 11


In [3]:
print(
    classification_report(
        y_test,
        rf_pred,
        digits=4,
        zero_division=0
    )
)

labels = sorted(y.unique())

rf_confusion = pd.DataFrame(
    confusion_matrix(y_test, rf_pred, labels=labels),
    index=labels,
    columns=labels
)

rf_confusion

              precision    recall  f1-score   support

      benign     0.9988    0.9964    0.9976    558225
  bruteforce     0.9892    0.9919    0.9906      2222
         dos     0.9945    0.9973    0.9959     94147
    portscan     0.9886    0.9991    0.9938     27185
         web     0.1577    0.3461    0.2167       627

    accuracy                         0.9961    682406
   macro avg     0.8258    0.8662    0.8389    682406
weighted avg     0.9970    0.9961    0.9965    682406



,benign,bruteforce,dos,portscan,web
benign,556240,24,506,306,1149
bruteforce,18,2204,0,0,0
dos,239,0,93894,7,7
portscan,10,0,11,27161,3
web,404,0,5,1,217


In [4]:
is_benign = y_test == "benign"

false_alarms = (
    (rf_pred != "benign") & is_benign
).sum()

far = false_alarms / is_benign.sum()

is_attack = ~is_benign

detected = (
    (rf_pred != "benign") & is_attack
).sum()

attack_detection_rate = detected / is_attack.sum()

print(
    f"False alarm rate: {far:.4%} "
    f"({false_alarms} of {is_benign.sum()} benign test flows)"
)

print(
    f"Attack detection rate: {attack_detection_rate:.4%} "
    f"({detected} of {is_attack.sum()} attack test flows)"
)

print("Under 1% false alarm target:", far < 0.01)

False alarm rate: 0.3556% (1985 of 558225 benign test flows)
Attack detection rate: 99.4597% (123510 of 124181 attack test flows)
Under 1% false alarm target: True


In [5]:
rf_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns,
    name="importance"
).sort_values(ascending=False)

rf_importance

dst_port       2.037638e-01
fwd_bytes      1.360025e-01
bwd_bytes      1.327224e-01
bwd_pkts       1.136697e-01
duration_us    1.106901e-01
pkts_per_s     1.042372e-01
bytes_per_s    9.716416e-02
fwd_pkts       9.669998e-02
fin_count      2.767418e-03
syn_count      2.281875e-03
rst_count      9.240246e-07
Name: importance, dtype: float64

In [6]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

rf_path = MODEL_DIR / "rf_baseline_v0.joblib"

joblib.dump(
    {
        "model": rf,
        "features": list(X.columns)
    },
    rf_path
)

print("Random Forest model saved to:", rf_path.resolve())

Random Forest model saved to: C:\Users\Harman\Lumen-IDS\models\rf_baseline_v0.joblib


In [7]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Convert class names into numeric labels for XGBoost
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

sample_weights = compute_sample_weight(
    "balanced",
    y_train
)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    eval_metric="mlogloss"
)

xgb.fit(
    X_train,
    y_train_encoded,
    sample_weight=sample_weights
)

xgb_pred_encoded = xgb.predict(X_test).astype(int)

xgb_pred = le.inverse_transform(xgb_pred_encoded)

print("XGBoost training and prediction complete.")

XGBoost training and prediction complete.


In [8]:
print(
    classification_report(
        y_test,
        xgb_pred,
        digits=4,
        zero_division=0
    )
)

xgb_confusion = pd.DataFrame(
    confusion_matrix(y_test, xgb_pred, labels=labels),
    index=labels,
    columns=labels
)

display(xgb_confusion)

ben = y_test == "benign"

xgb_false_alarms = (
    (xgb_pred != "benign") & ben
).sum()

xgb_far = xgb_false_alarms / ben.sum()

attacks = ~ben

xgb_detected = (
    (xgb_pred != "benign") & attacks
).sum()

xgb_detection_rate = xgb_detected / attacks.sum()

print(
    f"XGBoost false alarm rate: {xgb_far:.4%} "
    f"({xgb_false_alarms} of {ben.sum()} benign test flows)"
)

print(f"XGBoost attack detection rate: {xgb_detection_rate:.4%}")

              precision    recall  f1-score   support

      benign     0.9998    0.9897    0.9947    558225
  bruteforce     0.9754    0.9977    0.9864      2222
         dos     0.9929    0.9984    0.9956     94147
    portscan     0.9881    0.9993    0.9937     27185
         web     0.1144    0.9761    0.2049       627

    accuracy                         0.9913    682406
   macro avg     0.8141    0.9923    0.8351    682406
weighted avg     0.9975    0.9913    0.9941    682406



,benign,bruteforce,dos,portscan,web
benign,552496,56,662,319,4692
bruteforce,4,2217,0,0,1
dos,105,0,93995,8,39
portscan,5,0,9,27167,4
web,10,0,4,1,612


XGBoost false alarm rate: 1.0263% (5729 of 558225 benign test flows)
XGBoost attack detection rate: 99.9001%


In [9]:
comparison = pd.DataFrame({
    "model": ["Random Forest", "XGBoost"],
    "false_alarm_rate": [far, xgb_far],
    "attack_detection_rate": [
        attack_detection_rate,
        xgb_detection_rate
    ]
})

comparison

,model,false_alarm_rate,attack_detection_rate
0,Random Forest,0.003556,0.994597
1,XGBoost,0.010263,0.999001


In [10]:
xgb_path = MODEL_DIR / "xgb_baseline_v0.joblib"

joblib.dump(
    {
        "model": xgb,
        "label_encoder": le,
        "features": list(X.columns)
    },
    xgb_path
)

print("XGBoost model saved to:", xgb_path.resolve())

XGBoost model saved to: C:\Users\Harman\Lumen-IDS\models\xgb_baseline_v0.joblib


In [5]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Load the prepared CIC dataset
DATA_PATH = Path("../data/processed/cic_common.pkl")
feats = pd.read_pickle(DATA_PATH)

# Define the 11 original features
EXPECTED = [
    "dst_port", "duration_us", "fwd_pkts", "bwd_pkts",
    "fwd_bytes", "bwd_bytes", "bytes_per_s", "pkts_per_s",
    "syn_count", "rst_count", "fin_count"
]

# Separate features and target
X = feats[EXPECTED].copy()
y = feats["family"].copy()

# Recreate the same train/test split as the original notebook
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", X_train.columns.tolist())

Training rows: 1592280
Testing rows: 682406
Features: ['dst_port', 'duration_us', 'fwd_pkts', 'bwd_pkts', 'fwd_bytes', 'bwd_bytes', 'bytes_per_s', 'pkts_per_s', 'syn_count', 'rst_count', 'fin_count']


In [11]:
import joblib
from pathlib import Path

Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": rf,
        "features": list(X.columns)
    },
    "../models/rf_baseline_v0.joblib"
)

print("Model saved successfully.")

Model saved successfully.


In [6]:
# Feature-set experiment: A, B and C
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd
import joblib

# A: All 11 features
features_A = [
    "dst_port", "duration_us", "fwd_pkts", "bwd_pkts",
    "fwd_bytes", "bwd_bytes", "bytes_per_s", "pkts_per_s",
    "syn_count", "rst_count", "fin_count"
]

# B: Remove unreliable flag-count features
flag_features = ["syn_count", "rst_count", "fin_count"]
features_B = [f for f in features_A if f not in flag_features]

# C: Five portable features (no flags or timing/rate features)
features_C = [
    "dst_port",
    "fwd_pkts",
    "bwd_pkts",
    "fwd_bytes",
    "bwd_bytes",
]

feature_sets = {
    "A_all11": features_A,
    "B_no_flags": features_B,
    "C_portable5": features_C,
}

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

experiment_results = []
experiment_reports = {}

for name, columns in feature_sets.items():
    print(f"\n{'=' * 55}")
    print(f"Training model {name}: {len(columns)} features")

    model = RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train[columns], y_train)
    predictions = model.predict(X_test[columns])

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division=0,
    )
    experiment_reports[name] = report

    benign_mask = y_test == "benign"
    false_alarms = (
        (predictions != "benign") & benign_mask
    ).sum()
    false_alarm_rate = false_alarms / benign_mask.sum()

    attack_mask = ~benign_mask
    attack_detected = (
        (predictions != "benign") & attack_mask
    ).sum()
    attack_detection_rate = attack_detected / attack_mask.sum()

    experiment_results.append({
        "model": name,
        "feature_count": len(columns),
        "false_alarm_rate": false_alarm_rate,
        "attack_detection_rate": attack_detection_rate,
        "benign_recall": report["benign"]["recall"],
        "portscan_recall": report.get("portscan", {}).get("recall", 0),
        "dos_recall": report.get("dos", {}).get("recall", 0),
        "bruteforce_recall": report.get("bruteforce", {}).get("recall", 0),
        "web_recall": report.get("web", {}).get("recall", 0),
    })

    model_path = MODEL_DIR / f"rf_{name}.joblib"
    joblib.dump(
        {
            "model": model,
            "features": columns,
        },
        model_path,
    )

    print(f"False alarm rate: {false_alarm_rate:.4%}")
    print(f"Attack detection rate: {attack_detection_rate:.4%}")
    print("Saved:", model_path.resolve())

results_df = pd.DataFrame(experiment_results)

print("\nFINAL FEATURE-SET COMPARISON")
display(
    results_df.style.format({
        "false_alarm_rate": "{:.4%}",
        "attack_detection_rate": "{:.4%}",
        "benign_recall": "{:.4%}",
        "portscan_recall": "{:.4%}",
        "dos_recall": "{:.4%}",
        "bruteforce_recall": "{:.4%}",
        "web_recall": "{:.4%}",
    })
)



Training model A_all11: 11 features
False alarm rate: 0.3556%
Attack detection rate: 99.4597%
Saved: C:\Users\Harman\Lumen-IDS\models\rf_A_all11.joblib

Training model B_no_flags: 8 features
False alarm rate: 0.3339%
Attack detection rate: 99.4299%
Saved: C:\Users\Harman\Lumen-IDS\models\rf_B_no_flags.joblib

Training model C_portable5: 5 features
False alarm rate: 2.2204%
Attack detection rate: 99.9316%
Saved: C:\Users\Harman\Lumen-IDS\models\rf_C_portable5.joblib

FINAL FEATURE-SET COMPARISON


,model,feature_count,false_alarm_rate,attack_detection_rate,benign_recall,portscan_recall,dos_recall,bruteforce_recall,web_recall
0,A_all11,11,0.3556%,99.4597%,99.6444%,99.9117%,99.7313%,99.1899%,34.6093%
1,B_no_flags,8,0.3339%,99.4299%,99.6661%,99.9117%,99.7122%,99.2349%,31.1005%
2,C_portable5,5,2.2204%,99.9316%,97.7796%,99.8124%,99.9150%,99.8200%,98.2456%


In [7]:
D = ["dst_port", "fwd_pkts", "bwd_pkts", "fwd_bytes"]
m = RandomForestClassifier(n_estimators=100, min_samples_leaf=2,
                           class_weight="balanced_subsample", n_jobs=-1, random_state=42)
m.fit(X_train[D], y_train)
p = m.predict(X_test[D])
ben = (y_test == "benign")
print("CIC false alarm rate:", ((p != "benign") & ben).sum() / ben.sum())
print({c: round(((p == c) & (y_test == c)).sum() / (y_test == c).sum(), 4)
       for c in ["bruteforce", "dos", "portscan", "web"]})
joblib.dump({"model": m, "features": D}, "../models/rf_D_overlap4.joblib")

CIC false alarm rate: 0.02667024945138609
{'bruteforce': np.float64(0.9986), 'dos': np.float64(0.993), 'portscan': np.float64(0.9974), 'web': np.float64(0.9809)}


['../models/rf_D_overlap4.joblib']